In [52]:
import pandas as pd
import numpy as np
import importlib
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform

import plotly.graph_objects as go
import plotly.express as px

import irina.utility_functions as uf

In [56]:
year = 2022
mode = "individual"
df_cs = pd.read_csv(uf.PATH+f"df_country_subfield/{mode}_{year}.csv", index_col=0)

In [57]:
df_cf = (
    df_cs
    .T
    .assign(subfield_id=lambda df: df.index.astype(int))
    .merge(uf.df_topics[["subfield_id", "field_id"]].drop_duplicates(),
           left_on="subfield_id", right_on="subfield_id", how="left")
    .assign(field_str=lambda df: df.field_id.astype(str))
    .groupby("field_str")
    .sum()
    .replace(0, np.nan)
    .drop(["subfield_id", "field_id"], axis=1)
    .T
)
df_cf_prob = df_cf.div(df_cf.sum(axis=1), axis=0)

In [58]:
df_cf_prob

field_str,11,12,13,14,15,16,17,18,19,20,...,27,28,29,30,31,32,33,34,35,36
AD,NaN,0.200000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.100000,NaN,NaN,NaN,NaN,NaN,0.400000,NaN,NaN,NaN
AE,0.018210,0.023762,0.026427,0.050189,0.005330,0.008217,0.117255,0.011992,0.007328,0.023318,...,0.156340,0.009993,0.004664,0.003997,0.018210,0.029536,0.110593,0.000444,0.007995,0.017322
AF,0.029762,0.059524,0.005952,0.065476,NaN,0.005952,0.044643,0.002976,0.008929,0.041667,...,0.190476,0.005952,0.017857,NaN,0.014881,0.047619,0.321429,NaN,0.020833,0.011905
AG,NaN,0.013699,0.027397,0.013699,NaN,NaN,0.013699,NaN,NaN,NaN,...,0.780822,0.027397,NaN,NaN,0.013699,0.041096,0.027397,NaN,0.027397,0.013699
AL,0.042969,0.058594,0.018229,0.058594,NaN,0.002604,0.063802,0.014323,0.019531,0.080729,...,0.199219,0.005208,0.005208,NaN,0.006510,0.032552,0.223958,NaN,0.003906,0.023438
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
WS,0.105263,0.052632,NaN,0.052632,NaN,NaN,NaN,NaN,0.157895,NaN,...,0.105263,NaN,NaN,NaN,NaN,0.052632,0.263158,NaN,NaN,0.105263
YE,0.038991,0.034404,0.036697,0.036697,0.002294,0.016055,0.103211,0.016055,0.011468,0.034404,...,0.183486,NaN,0.016055,0.016055,0.032110,0.025229,0.139908,NaN,0.018349,0.020642
ZA,0.061678,0.043670,0.032415,0.053374,0.002251,0.010155,0.046071,0.014957,0.011655,0.036116,...,0.137812,0.006153,0.007553,0.002501,0.012756,0.040218,0.222400,0.001151,0.002951,0.036666
ZM,0.051864,0.017828,0.019449,0.090762,NaN,NaN,0.051864,0.034036,0.001621,0.071313,...,0.228525,0.001621,0.009724,NaN,0.008104,0.034036,0.181524,NaN,0.003241,0.056726


In [59]:
uf.df_topics.query("field_id == 22")[["subfield_name", "field_name", "domain_name"]].drop_duplicates()

,subfield_name,field_name,domain_name
17,Electrical and Electronic Engineering,Engineering,Physical Sciences
32,Civil and Structural Engineering,Engineering,Physical Sciences
39,Control and Systems Engineering,Engineering,Physical Sciences
58,Biomedical Engineering,Engineering,Physical Sciences
68,Aerospace Engineering,Engineering,Physical Sciences
120,Building and Construction,Engineering,Physical Sciences
160,Mechanics of Materials,Engineering,Physical Sciences
172,Computational Mechanics,Engineering,Physical Sciences
187,Mechanical Engineering,Engineering,Physical Sciences
369,"Safety, Risk, Reliability and Quality",Engineering,Physical Sciences


In [60]:
y_labels = df_cf.sum(axis=1).sort_values(ascending=False).index.to_list()
uf.plotly_heatmap(df_cf_prob.loc[y_labels],
                  x_labels=df_cf_prob.columns,
                  y_labels=y_labels,
                  colorscale="non")

In [61]:
importlib.reload(uf)

<module 'irina.utility_functions' from '/Users/irinavorobeva/PycharmProjects/geoTopics/irina/utility_functions.py'>

In [62]:
field_tree = uf.get_field_tree(uf.df_topics)
df_dist = uf.get_w1_distances(field_tree, df_cf_prob, subfield_mode=False)

In [63]:
df_dist

,AD,AE,AF,AG,AL,AM,AO,AR,AT,AU,...,VE,VG,VI,VN,VU,WS,YE,ZA,ZM,ZW
AD,0.000000,0.383839,0.220346,0.784184,0.236648,0.235746,0.386085,0.231654,0.314560,0.375775,...,0.289619,0.713188,0.226203,0.358217,0.659091,0.239713,0.348707,0.229603,0.321689,0.146505
AE,0.383839,0.000000,0.346423,0.638668,0.273283,0.214536,0.367920,0.300512,0.136148,0.240550,...,0.318109,0.507544,0.406845,0.072246,0.529506,0.266873,0.147116,0.210364,0.319350,0.343780
AF,0.220346,0.346423,0.000000,0.590186,0.087375,0.281034,0.191829,0.160536,0.253560,0.254671,...,0.102649,0.826139,0.160316,0.382683,0.483225,0.172548,0.257057,0.187218,0.123271,0.146740
AG,0.784184,0.638668,0.590186,0.000000,0.595787,0.726875,0.427758,0.661057,0.612568,0.495413,...,0.584449,0.964458,0.710497,0.703849,0.402242,0.627188,0.589410,0.640760,0.532713,0.724514
AL,0.236648,0.273283,0.087375,0.595787,0.000000,0.204781,0.199484,0.089653,0.175407,0.185741,...,0.075800,0.768661,0.188997,0.302294,0.472775,0.098933,0.187932,0.108620,0.091311,0.140198
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
WS,0.239713,0.266873,0.172548,0.627188,0.098933,0.163066,0.229836,0.105367,0.143232,0.164031,...,0.099578,0.731900,0.195609,0.267959,0.453349,0.000000,0.173456,0.063935,0.128000,0.148383
YE,0.348707,0.147116,0.257057,0.589410,0.187932,0.154921,0.281876,0.216818,0.052221,0.108185,...,0.184520,0.649841,0.323125,0.151664,0.431610,0.173456,0.000000,0.133021,0.210061,0.268423
ZA,0.229603,0.210364,0.187218,0.640760,0.108620,0.105893,0.245874,0.095212,0.110531,0.157074,...,0.114078,0.705276,0.204147,0.211959,0.477017,0.063935,0.133021,0.000000,0.139132,0.139549
ZM,0.321689,0.319350,0.123271,0.532713,0.091311,0.239749,0.138467,0.152302,0.207831,0.148667,...,0.067052,0.813436,0.210212,0.328789,0.395683,0.128000,0.210061,0.139132,0.000000,0.205099


In [64]:
labels = df_cf.sum(axis=1).sort_values(ascending=False).index.to_list()
uf.plotly_heatmap(df_dist.loc[labels, labels],
                  x_labels=labels,
                  y_labels=labels,
                  colorscale="non")

In [65]:
# Example: distance matrix (symmetric, zeros on diagonal)
D = df_dist.fillna(0).values

# Convert to condensed form (required by linkage)
condensed_D = squareform(D)

# Perform hierarchical clustering
# method can be: 'single', 'complete', 'average', 'ward'
Z = linkage(condensed_D, method='average')


In [66]:
# Assign clusters by specifying a max distance threshold
labels = fcluster(Z, t=0.2, criterion='distance')
order = np.argsort(labels)
# print("Cluster labels:", labels)

In [67]:
labels = np.array(labels)

order = np.argsort(labels)
D_ordered = D[np.ix_(order, order)]

names = df_dist.index.to_numpy()
names_ordered = [uf.id2name_country[name] for name in names[order]]
labels_ordered = labels[order]

In [68]:
fig = go.Figure(
    data=go.Heatmap(
        z=D_ordered,
        x=names_ordered,
        y=names_ordered,
        colorscale="Viridis",
        colorbar=dict(title="Distance"),
        hovertemplate="Row: %{y}<br>Col: %{x}<br>Dist: %{z:.3f}<extra></extra>"
    )
)

fig.update_layout(
    title="Distance matrix reordered by cluster labels",
    xaxis=dict(
        tickangle=45,
        tickfont=dict(size=8),
        automargin=True
    ),
    yaxis=dict(
        tickfont=dict(size=8),
        automargin=True
    ),
    width=900,
    height=900
)
# cluster boundaries
changes = np.where(np.diff(labels_ordered) != 0)[0] + 1

for c in changes:
    fig.add_shape(
        type="line",
        x0=-0.5, x1=len(names_ordered)-0.5,
        y0=c-0.5, y1=c-0.5,
        line=dict(color="white", width=1)
    )
    fig.add_shape(
        type="line",
        x0=c-0.5, x1=c-0.5,
        y0=-0.5, y1=len(names_ordered)-0.5,
        line=dict(color="white", width=1)
    )

fig.show()


In [69]:
df_map = (
    df_dist
    .merge(uf.df_country[["alpha-2", "alpha-3", "name", "region", "sub-region"]], left_index=True, right_on="alpha-2", how="left")
    [["alpha-2", "alpha-3", "name", "sub-region", "region"]]
    .assign(cluster=labels,
            cluster_str = lambda df: df["cluster"].astype(str))
    .rename(columns={"alpha-3": "country", "alpha-2": "country2"})
)
df_map.sample(5)

,country2,country,name,sub-region,region,cluster,cluster_str
198,SC,SYC,Seychelles,Sub-Saharan Africa,Africa,3,3
54,CI,CIV,Côte d'Ivoire,Sub-Saharan Africa,Africa,13,13
237,UY,URY,Uruguay,Latin America and the Caribbean,Americas,16,16
209,ES,ESP,Spain,Southern Europe,Europe,16,16
119,KR,KOR,"Korea, Republic of",Eastern Asia,Asia,14,14


In [70]:
fig = px.choropleth(
    df_map,
    locations="country",
    color="cluster_str",               # use the categorical version
    locationmode="ISO-3",
    color_discrete_sequence=px.colors.qualitative.Set3,  # discrete color palette
    title=f"Country clusters based on distance matrix, {mode[:-1]}, {year}"
)

fig.update_layout(
    geo=dict(
        showframe=False,
        showcoastlines=True,
        projection_type="natural earth"
    ),
    height=600,
)

fig.show()